In [2]:
import pickle

with open('gradient_test_exponax_nu0.01_nsamples1.pkl', 'rb') as f: 
    data = pickle.load(f)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
from tqdm import tqdm  # for progress bar
import os

def calculate_global_limits(data):
    """
    Calculate global min/max values across all configurations and steps
    Returns: (global_min, global_max)
    """
    all_values = []
    
    # Iterate through all configurations
    for config_key in data.keys():
        if not isinstance(config_key, tuple):
            continue
            
        try:
            step_data = data[config_key]['step_update']
        except KeyError:
            continue
            
        # Iterate through all steps
        for step_key in step_data.keys():
            if not step_key.startswith('step_'):
                continue
                
            try:
                values = step_data[step_key]['values']['a_values']
                # Collect all relevant arrays
                all_values.append(values['a'])
                all_values.append(values['a_with'])
                all_values.append(values['a_without'])  # Original key
                all_values.append(values['a_approximate'])
            except KeyError:
                continue
    
    if not all_values:
        return 0.99, 1.001  # Default fallback
    
    # Convert list of arrays to single array
    all_values = np.concatenate(all_values)
    
    # Calculate padding (5% of range)
    value_range = np.max(all_values) - np.min(all_values)
    padding = value_range * 0.05
    
    global_min = np.min(all_values) - padding
    global_max = np.max(all_values) + padding
    
    return float(global_min), float(global_max)

def create_adversarial_evolution_gif(data, config_key, output_dir="gifs"):
    os.makedirs(output_dir, exist_ok=True)
    
    # 强制获取数据
    step_data = data[config_key]['step_update']
    final_values = data[config_key]['final_values']
    
    # 按步骤顺序处理
    valid_steps = sorted([k for k in step_data.keys() if k.startswith('step_')],
                        key=lambda x: int(x.split('_')[1]))[1:]
    
    num_steps = len(valid_steps)
    if num_steps == 0:
        raise ValueError(f"No valid steps found for {config_key}")
    
    # ==================== 1. 预计算所有数据 ====================
    all_values = {
        'a': [], 
        'a_with': [], 
        'a_without': [],
        'a_approximate': []
    }
    losses = {
        'with': [], 
        'without': [],
        'approximate': []
    }
    # 新增：存储每步的 G 和 g 数据
    all_G_values = {
        'G(a)': [],
        'G(a_with)': [],
        'G(a_without)': [],
        'G(a_approximate)': []
    }
    all_g_values = {
        'g(a)': [],
        'g(a_with)': [],
        'g(a_without)': [],
        'g(a_approximate)': []
    }

    for step_key in valid_steps:
        step = step_data[step_key]
        vals = step['values']
        # 存储 a 值
        for k in all_values.keys():
            all_values[k].append(vals['a_values'][k])
        # 新增：存储 G 和 g 值
        for k in all_G_values.keys():
            all_G_values[k].append(vals['G_values'][k])
        for k in all_g_values.keys():
            all_g_values[k].append(vals['g_values'][k])

    # 存储 loss（保持原样）
    for step_key in ['step_0']+valid_steps:   
        step = step_data[step_key]
        loss = step['loss_metrics']
        losses['with'].append(loss['with_solver'])
        losses['without'].append(loss['without_solver'])
        losses['approximate'].append(loss['approximate'])

    # 转换为 numpy 数组
    for k in all_values:
        all_values[k] = np.array(all_values[k])
    for k in all_G_values:  # 新增
        all_G_values[k] = np.array(all_G_values[k])
    for k in all_g_values:  # 新增
        all_g_values[k] = np.array(all_g_values[k])
    
    # ==================== 2. 计算坐标范围 ====================
    adv_ymin = np.min([np.min(arr) for arr in all_values.values()])
    adv_ymax = np.max([np.max(arr) for arr in all_values.values()])
    padding = (adv_ymax - adv_ymin) * 0.1
    adv_ymin -= padding
    adv_ymax += padding

    # G/g 值范围（改为从 all_G_values 和 all_g_values 计算）
    g_g_ylim = (
        min(np.min(arr) for arr in list(all_G_values.values()) + list(all_g_values.values())),
        max(np.max(arr) for arr in list(all_G_values.values()) + list(all_g_values.values()))
    )
    padding = (g_g_ylim[1] - g_g_ylim[0]) * 0.1
    g_g_ylim = (g_g_ylim[0] - padding, g_g_ylim[1] + padding)

    all_loss_values = np.concatenate([
        losses['with'],
        losses['without'],
        losses['approximate']
    ])

    # 计算最小/最大值（忽略可能的NaN/Inf）
    loss_ymin = np.nanmin(all_loss_values)
    loss_ymax = np.nanmax(all_loss_values)

    # 处理所有损失值相同的情况（如全为0）
    if loss_ymin == loss_ymax:
        loss_ymin -= 0.1
        loss_ymax += 0.1
    else:
        # 添加10%的缓冲区域
        padding = (loss_ymax - loss_ymin) * 0.1
        loss_ymin -= padding
        loss_ymax += padding

    # 确保最小值不为负（如果损失函数非负）
    loss_ymin = max(loss_ymin, 0) if (all_loss_values >= 0).all() else loss_ymin
    
    # ==================== 3. 创建图形 ====================
    fig = plt.figure(figsize=(18, 12))  # 高度从 18 调整为 12（因为行数减少）
    gs = fig.add_gridspec(2, 3)  # 改为 2 行 3 列
    x_grid = np.linspace(0, 1, 1024)
    
    ax1 = fig.add_subplot(gs[0, 0])  # 对抗演化
    ax2 = fig.add_subplot(gs[1, 0])  # 损失变化
    ax3 = fig.add_subplot(gs[0, 1])  # 原始G vs g
    ax4 = fig.add_subplot(gs[0, 2])  # With Solver G vs g
    ax5 = fig.add_subplot(gs[1, 1])  # Without Solver G vs g
    ax6 = fig.add_subplot(gs[1, 2])  # Approximate G vs g
    
    plt.tight_layout(pad=5.0)
    
    # ==================== 4. 初始化图形元素 ====================
    lines = {
        'original': ax1.plot(x_grid, all_values['a'][0], 'b', linewidth=1.5, label='Original (a)', alpha=0.5)[0],
        'with': ax1.plot(x_grid, all_values['a_with'][0], 'r', linewidth=1.5, label='With Solver', alpha=0.5)[0],
        'detach': ax1.plot(x_grid, all_values['a_without'][0], 'k', linewidth=1.5, label='Detach', alpha=0.5)[0],
        'approximate': ax1.plot(x_grid, all_values['a_approximate'][0], 'g', linewidth=1.5, label='Approximate', alpha=0.5)[0]
    }
    ax1.set_ylim(adv_ymin, adv_ymax)
    ax1.set_xlabel('Position')
    ax1.set_ylabel('Value')
    ax1.grid(True, alpha=0.3)
    ax1.set_title("Perturbed Inputs vs. Original")
    ax1.legend()
    
    loss_lines = {
        'with': ax2.plot([0], [losses['with'][0]], 'r', linewidth=2, label='With Solver', marker='o', markersize=2)[0],
        'detach': ax2.plot([0], [losses['without'][0]], 'k', linewidth=2, label='Detach', marker='o', markersize=2)[0],
        'approximate': ax2.plot([0], [losses['approximate'][0]], 'g', linewidth=2, label='Approximate', marker='o', markersize=2)[0]
    }
    print(losses.keys())  # 应输出: dict_keys(['with', 'without', 'approximate'])
    print(len(losses['with']), len(losses['without']), len(losses['approximate']))
    ax2.set_xlim(0, num_steps)
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Loss')
    ax2.set_ylim(loss_ymin, loss_ymax)
    ax2.grid(True, alpha=0.3)
    ax2.set_title("Loss Progression of Perturbation Methods")
    ax2.legend()
    
    def init_g_g_plot(ax, title):
        ax.set_xlim(0, 1)
        ax.set_ylim(*g_g_ylim)
        ax.set_xlabel('Position')
        ax.set_ylabel('Value')
        ax.grid(True, alpha=0.3)
        G_line = ax.plot([], [], 'b', linewidth=2, label='DeepONet')[0]
        g_line = ax.plot([], [], 'r', linewidth=2, label='PDE Solver')[0]
        title_obj = ax.set_title(title)
        ax.legend()
        return {'G': G_line, 'g': g_line, 'title': title_obj}
    
    g_g_plots = {
        'original': init_g_g_plot(ax3, 'Original: DeepONet output vs PDE Solver output'),
        'with solver': init_g_g_plot(ax4, 'With Solver: DeepONet output vs PDE Solver output'),
        'detach': init_g_g_plot(ax5, 'Detach: DeepONet output vs PDE Solver output'),
        'approximate': init_g_g_plot(ax6, 'Approximate: DeepONet output vs PDE Solver output')
    }
    
    # ==================== 5. 动画更新函数 ====================
    def calculate_metrics(G, g):
        diff = G - g
        rmse = np.sqrt(np.mean(diff**2))
        mae = np.mean(np.abs(diff))
        return rmse, mae
    
    def update(frame):
        actual_step_index = frame  # 从1开始
        
        # 更新对抗演化曲线
        lines['original'].set_ydata(all_values['a'][actual_step_index])
        lines['with'].set_ydata(all_values['a_with'][actual_step_index])
        lines['detach'].set_ydata(all_values['a_without'][actual_step_index])
        lines['approximate'].set_ydata(all_values['a_approximate'][actual_step_index])
        
        # 更新损失曲线
        x = range(0, actual_step_index)
        loss_lines['with'].set_data(x, losses['with'][0:actual_step_index])
        loss_lines['detach'].set_data(x, losses['without'][0:actual_step_index])
        loss_lines['approximate'].set_data(x, losses['approximate'][0:actual_step_index])
        
        # 更新 G vs g 图（从预计算的中间数据获取）
        g_g_mappings = {
            'original': ('', ''),
            'with solver': ('_with', '_with'),
            'detach': ('_without', '_without'),
            'approximate': ('_approximate', '_approximate')
        }
        
        for method, (g_suffix, G_suffix) in g_g_mappings.items():
            G_key = f"G(a{G_suffix})"
            g_key = f"g(a{g_suffix})"
            
            # 关键修改：从预计算的中间数据获取当前帧的值
            G_data = all_G_values[G_key][frame]
            g_data = all_g_values[g_key][frame]
            
            plot = g_g_plots[method]
            plot['G'].set_data(x_grid, G_data)
            plot['g'].set_data(x_grid, g_data)
            
            rmse, mae = calculate_metrics(G_data, g_data)
            plot['title'].set_text(f"{method.capitalize()}: DeepONet output vs PDE Solver output\nRMSE: {rmse:.4f}, MAE: {mae:.4f}")
        
        # 更新标题
        step_key = valid_steps[actual_step_index]
        boundary_info = step_data[step_key].get('reaches_boundary', {})
        boundary_text = " | ".join([f"{k.replace('_', ' ')}: {v}" for k, v in boundary_info.items()])
        fig.suptitle(f"Config: {config_key} | Step {actual_step_index} | Reaches Boundary: {boundary_text}", y=1.00, fontsize=12)
        
        return (list(lines.values()) + 
                list(loss_lines.values()) + 
                [v['G'] for v in g_g_plots.values()] + 
                [v['g'] for v in g_g_plots.values()] + 
                [v['title'] for v in g_g_plots.values()] + 
                [fig._suptitle])
    
    # ==================== 6. 创建并保存动画 ====================
    if isinstance(config_key, tuple):
        filename = "_".join(str(x) for x in config_key) + ".gif"
    else:
        filename = str(config_key) + ".gif"
    filepath = os.path.join(output_dir, filename)
    
    ani = animation.FuncAnimation(
        fig, update, 
        frames=range(0, num_steps),  # 跳过第一帧
        blit=False, 
        interval=300, 
    )
    
    with tqdm(total=num_steps, desc=f"Saving {filename}") as pbar:
        ani.save(
            filepath,
            writer=animation.PillowWriter(fps=3),
            progress_callback=lambda i, n: pbar.update(1),
            dpi=100
        )
    
    plt.close(fig)
    print(f"Successfully saved: {filepath}")

def process_all_configs(data):
    """Process all configuration keys found in the data"""
    config_keys = [k for k in data.keys() if isinstance(k, tuple)]
    
    print(f"Found {len(config_keys)} configurations to process")
    for key in config_keys:
        create_adversarial_evolution_gif(data, key)

# Usage:
process_all_configs(data)

Found 2 configurations to process
dict_keys(['with', 'without', 'approximate'])
101 101 101


Saving norm_inf_index_0_numsteps_100_epsilon_0.5_alpha_0.01000_sample_0.gif: 100%|██████████| 100/100 [00:57<00:00,  1.64it/s]